Start by importing the required packages.

In [162]:
import pandas as pd
!pip install geopy

Load in the CSV file and look and the column names and datatypes.

In [164]:
atp2023 = pd.read_csv('atp_matches_2023.csv')
atp2023.dtypes

tourney_id             object
tourney_name           object
surface                object
draw_size               int64
tourney_level          object
tourney_date            int64
match_num               int64
winner_id               int64
winner_seed           float64
winner_entry           object
winner_name            object
winner_hand            object
winner_ht             float64
winner_ioc             object
winner_age            float64
loser_id                int64
loser_seed            float64
loser_entry            object
loser_name             object
loser_hand             object
loser_ht              float64
loser_ioc              object
loser_age             float64
score                  object
best_of                 int64
round                  object
minutes               float64
w_ace                 float64
w_df                  float64
w_svpt                float64
w_1stIn               float64
w_1stWon              float64
w_2ndWon              float64
w_SvGms   

Drop unnecessary columns, change the date column to datetime, and list unique values in the tourney_name column.

In [166]:
atp2023 = atp2023[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]
atp2023.dtypes

tourney_id       object
tourney_name     object
surface          object
draw_size         int64
tourney_level    object
tourney_date      int64
dtype: object

In [167]:
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals', 'Davis Cup

Remove the values that contain "Davis Cup". These won't be used in the analysis.

In [169]:
atp2023 = atp2023[~atp2023['tourney_name'].str.contains('Davis Cup', case=False, na=False)]
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=obj

Remove "Masters" from any of the values. This will help find the locations.

In [171]:
atp2023["tourney_name"] = atp2023["tourney_name"].str.replace("masters", "", case=False, regex=True).str.strip()
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells', 'Miami', 'Estoril', 'Houston', 'Marrakech',
       'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka', 'Madrid',
       'Rome', 'Geneva', 'Lyon', 'Roland Garros', 's Hertogenbosch',
       'Stuttgart', 'Halle', "Queen's Club", 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos', 'Washington', 'Canada',
       'Cincinnati', 'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai',
       'Astana', 'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=object)

Change the tournament names without clear locations to be more specific.

In [173]:
atp2023["tourney_name"] = atp2023["tourney_name"].replace({
    "Australian Open": "Melbourne",
    "Roland Garros": "Paris",
    "Queen's Club": "London",
    "Canada": "Toronto",
    "Us Open": "New York",
    "Tour Finals": "Turin",
    "NextGen Finals": "Jeddah",
    "Cordoba": "Cordoba, Argentina",
    "Los Cabos": "Los Cabos, Mexico",
    "Santiago": "Santiago, Chile"
})
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Melbourne', 'Cordoba, Argentina', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai',
       'Santiago, Chile', 'Indian Wells', 'Miami', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka',
       'Madrid', 'Rome', 'Geneva', 'Lyon', 'Paris', 's Hertogenbosch',
       'Stuttgart', 'Halle', 'London', 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos, Mexico', 'Washington', 'Toronto',
       'Cincinnati', 'Winston-Salem', 'New York', 'Chengdu', 'Zhuhai',
       'Astana', 'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Metz', 'Sofia', 'Turin', 'Jeddah'],
      dtype=object)

Change "United Cup" to the locations where the event is held.

In [175]:
atp2023["tourney_name"] = atp2023["tourney_name"].apply(
    lambda x: ["Brisbane", "Sydney", "Perth"] if x == "United Cup" else [x]
)

atp2023 = atp2023.explode("tourney_name")
atp2023["tourney_name"].unique()

array(['Brisbane', 'Sydney', 'Perth', 'Adelaide 1', 'Pune', 'Auckland',
       'Adelaide 2', 'Melbourne', 'Cordoba, Argentina', 'Dallas',
       'Montpellier', 'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai',
       'Santiago, Chile', 'Indian Wells', 'Miami', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka',
       'Madrid', 'Rome', 'Geneva', 'Lyon', 'Paris', 's Hertogenbosch',
       'Stuttgart', 'Halle', 'London', 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos, Mexico', 'Washington', 'Toronto',
       'Cincinnati', 'Winston-Salem', 'New York', 'Chengdu', 'Zhuhai',
       'Astana', 'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Metz', 'Sofia', 'Turin', 'Jeddah'],
      dtype=object)

Change the tourney_date column to only include the year

In [177]:
atp2023

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date
0,2023-9900,Brisbane,Hard,18,A,20230102
0,2023-9900,Sydney,Hard,18,A,20230102
0,2023-9900,Perth,Hard,18,A,20230102
1,2023-9900,Brisbane,Hard,18,A,20230102
1,2023-9900,Sydney,Hard,18,A,20230102
...,...,...,...,...,...,...
2761,2023-7696,Jeddah,Hard,8,F,20231127
2762,2023-7696,Jeddah,Hard,8,F,20231127
2763,2023-7696,Jeddah,Hard,8,F,20231127
2764,2023-7696,Jeddah,Hard,8,F,20231127


Create a dataframe that contains a list of the different locations tournaments are held and the locations for each one.

In [179]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

geolocator = Nominatim(user_agent="atp_2023_analysis")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # Nominatim rate limit

unique_cities = atp2023["tourney_name"].unique()

location_data = {}
for city in unique_cities:
    location = geocode(city)
    if location:
        location_data[city] = (location.latitude, location.longitude)
    else:
        print(f"Could not geocode: {city}")  # flag any misses

coord_df = pd.DataFrame.from_dict(
    location_data, orient="index", columns=["latitude", "longitude"]
)
coord_df.index.name = "tourney_name"
coord_df = coord_df.reset_index()

date_map = (
    atp2023.drop_duplicates("tourney_name")
           .set_index("tourney_name")["tourney_date"]  
)
coord_df["tourney_date"] = coord_df["tourney_name"].map(date_map)

In [180]:
coord_df["tourney_date"] = pd.to_datetime(
    coord_df["tourney_date"].astype(str),
    format="mixed"
)
coord_df["tourney_date"] = coord_df["tourney_date"].dt.year
coord_df.dtypes

tourney_name     object
latitude        float64
longitude       float64
tourney_date      int32
dtype: object

In [181]:
coord_df.to_csv("atp_city_coords5.csv", index=False)

In [182]:
coord_df

,tourney_name,latitude,longitude,tourney_date
0,Brisbane,-27.468962,153.023501,2023
1,Sydney,-33.869844,151.208285,2023
2,Perth,-31.955897,115.860578,2023
3,Adelaide 1,-34.819459,138.504749,2023
4,Pune,18.521374,73.854507,2023
...,...,...,...,...
63,Vienna,48.208354,16.372504,2023
64,Metz,49.119696,6.176355,2023
65,Sofia,42.697703,23.321736,2023
66,Turin,45.067755,7.682489,2023


In [204]:
def clean_atp(year):
    df = pd.read_csv(f"atp_matches_{year}.csv")
    df = df[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]
    
    df = df[~df['tourney_name'].str.contains('Davis Cup|Laver Cup', case=False, na=False)]
    df["tourney_name"] = df["tourney_name"].str.replace("masters", "", case=False, regex=True).str.strip()
    
    df["tourney_name"] = df["tourney_name"].replace({
        "Australian Open": "Melbourne",
        "Roland Garros": "Paris",
        "Queen's Club": "London",
        "Canada": "Montreal" if year % 2 == 0 else "Toronto",
        "Us Open": "New York",
        "Tour Finals": "Turin",
        "NextGen Finals": "Jeddah",
        "Next Gen Finals": "Jeddah",
        "Cordoba": "Cordoba, Argentina",
        "Los Cabos": "Los Cabos, Mexico",
        "Santiago": "Santiago, Chile",
        "Paris Olympics": "Paris O"
    })
    
    df["tourney_name"] = df["tourney_name"].apply(
        lambda x: ["Brisbane", "Sydney", "Perth"] if x == "United Cup" and year == 2023
        else ["Sydney", "Perth"] if x == "United Cup"
        else [x]
    )
    df = df.explode("tourney_name")
    
    df["year"] = year
    return df

# Load all years and combine
years = [2023, 2024, 2025, 2026]
all_years = pd.concat([clean_atp(y) for y in years], ignore_index=True)

In [205]:
# Load existing coords
coord_df = pd.read_csv("atp_city_coords.csv")
known_cities = set(coord_df["tourney_name"])

# Find any new cities not in the existing lookup
new_cities = set(all_years["tourney_name"].unique()) - known_cities

if new_cities:
    geolocator = Nominatim(user_agent="atp_analysis")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    
    new_coords = {}
    for city in new_cities:
        location = geocode(city)
        if location:
            new_coords[city] = (location.latitude, location.longitude)
        else:
            print(f"Could not geocode: {city}")
    
    new_coord_df = pd.DataFrame.from_dict(new_coords, orient="index", columns=["latitude", "longitude"])
    new_coord_df.index.name = "tourney_name"
    new_coord_df = new_coord_df.reset_index()
    
    coord_df = pd.concat([coord_df, new_coord_df], ignore_index=True)
    coord_df.to_csv("atp_city_coords.csv", index=False)  # save updated lookup

# Merge and export
final_df = all_years.drop_duplicates(subset=["tourney_name", "year"])
final_df = final_df.merge(coord_df[["tourney_name", "latitude", "longitude"]], on="tourney_name", how="left")
final_df.to_csv("atp_locations_2023_2026.csv", index=False)